# 🏥 AI-Powered Medical Image Analysis
### Chest X-Ray Pneumonia Detection using Transfer Learning (MobileNetV2)

---
**Author:** Your Name  
**Dataset:** Chest X-Ray Images (Pneumonia) – Kaggle  
**Model:** MobileNetV2 (Transfer Learning)  
**Task:** Binary Classification – NORMAL vs PNEUMONIA  

---

## 📋 Table of Contents
1. Setup & Imports
2. Dataset Overview
3. Image Preprocessing
4. Model Architecture
5. Training
6. Evaluation
7. Single-Image Prediction
8. Grad-CAM Visualisation
9. Conclusions

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import *
from src.data_loader import get_generators
from src.preprocessing import preprocess_for_model, get_preprocessing_stages
from src.model import build_model, get_model_summary
from src.train import train
from src.evaluate import evaluate, plot_training_history
from src.predict import predict_single
from src.visualize import plot_preprocessing_stages, plot_gradcam, plot_dataset_samples

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Classes            : {CLASS_NAMES}')
print(f'Image size         : {IMG_HEIGHT}x{IMG_WIDTH}')

## 2. Dataset Overview

In [ ]:
# ─── If you have downloaded the dataset, update this path ───────────────
DATASET_PATH = os.path.join(RAW_DIR, 'chest_xray')

# Count images per split
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATASET_PATH, split)
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(split_dir, cls)
        count = len(os.listdir(cls_dir)) if os.path.exists(cls_dir) else 0
        print(f'{split:6s}  {cls:10s}  {count:5d} images')

In [ ]:
# Plot class distribution
import pandas as pd

data = []
for split in ['train', 'val', 'test']:
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(DATASET_PATH, split, cls)
        count = len(os.listdir(cls_dir)) if os.path.exists(cls_dir) else 0
        data.append({'Split': split, 'Class': cls, 'Count': count})

df = pd.DataFrame(data)
pivoted = df.pivot(index='Split', columns='Class', values='Count')

pivoted.plot(kind='bar', figsize=(9,5), color=['#2ecc71','#e74c3c'])
plt.title('Class Distribution per Split', fontsize=14, fontweight='bold')
plt.xlabel('Split')
plt.ylabel('Image count')
plt.xticks(rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Image Preprocessing

In [ ]:
# Show preprocessing stages for one image
# Replace with any image path from your dataset
sample_img_path = os.path.join(DATASET_PATH, 'train', 'PNEUMONIA',
                               os.listdir(os.path.join(DATASET_PATH,'train','PNEUMONIA'))[0])

stages = get_preprocessing_stages(sample_img_path)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ['Original', 'CLAHE Enhanced', 'Resized 224×224', 'Normalised [0,1]']
for ax, (name, img), title in zip(axes, stages.items(), titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('Preprocessing Pipeline', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Architecture

In [ ]:
model = build_model(fine_tune_at=0)
get_model_summary(model)

## 5. Training

In [ ]:
# ⚠ This cell trains the model. Takes ~15 min on GPU, ~1-2 hr on CPU.
# Skip if you already have a saved model.

histories = train(run_fine_tuning=True)
print('Training complete.')

## 6. Evaluation

In [ ]:
import json

# Load histories if not already in memory
hist_path = os.path.join(REPORTS_DIR, 'histories.json')
if os.path.exists(hist_path):
    with open(hist_path) as f:
        histories = json.load(f)

metrics = evaluate(histories=histories)
print(f"\nTest Accuracy : {metrics['test_accuracy']*100:.2f}%")
print(f"F1-macro      : {metrics['f1_macro']:.4f}")

In [ ]:
# Display saved graphs inline
from IPython.display import Image as IPImage, display

for fname in ['training_history.png', 'confusion_matrix.png', 'roc_curve.png']:
    fpath = os.path.join(GRAPHS_DIR, fname)
    if os.path.exists(fpath):
        print(f'\n── {fname} ──')
        display(IPImage(filename=fpath, width=800))

## 7. Single-Image Prediction

In [ ]:
test_image = os.path.join(DATASET_PATH, 'test', 'PNEUMONIA',
             os.listdir(os.path.join(DATASET_PATH,'test','PNEUMONIA'))[0])

result = predict_single(test_image)

# Display annotated image
basename = os.path.splitext(os.path.basename(test_image))[0]
pred_path = os.path.join(PREDICTIONS_DIR, f'pred_{basename}.png')
if os.path.exists(pred_path):
    display(IPImage(filename=pred_path, width=400))

## 8. Grad-CAM Visualisation

In [ ]:
from src.model import load_model as load_saved_model

saved_model = load_saved_model()
plot_gradcam(test_image, saved_model, label=result['label'])

basename = os.path.splitext(os.path.basename(test_image))[0]
gcam_path = os.path.join(PREDICTIONS_DIR, f'gradcam_{basename}.png')
if os.path.exists(gcam_path):
    display(IPImage(filename=gcam_path, width=800))

## 9. Conclusions

| Metric | Value |
|--------|-------|
| Architecture | MobileNetV2 + custom head |
| Transfer Learning | ImageNet weights |
| Augmentation | Rotation, flip, zoom, shift |
| Task | Binary (NORMAL / PNEUMONIA) |

**Key learnings:**
- Transfer learning dramatically reduces training time on small medical datasets
- CLAHE pre-processing improves contrast in X-ray images
- Grad-CAM provides interpretability — critical for medical AI trust
- Class weighting handles the PNEUMONIA class imbalance in the Kaggle dataset

**Industry relevance:**
This pipeline mirrors real-world radiology AI assistants used in hospitals to flag abnormal chest X-rays for radiologist review, reducing manual workload by up to 40%.